# 🧹 Data Cleaning — Tweets SAVIA

Ce notebook nettoie les tweets en supprimant :

- les `@mentions`
- les `#hashtags`
- les URLs
- les emojis
- la ponctuation
- les espaces multiples

Puis il :
- filtre les comptes officiels (Free, Iliad, Assistance Freebox, etc.)
- exporte un fichier `tweets_cleaned_<timestamp>.csv` dans `data/silver/`
- enregistre une entrée qualité dans `savia/quality/monitoring_quality_log.csv`
    

In [2]:
# 📦 Imports
import pandas as pd
import re, os
from datetime import datetime
    

In [24]:
# 📂 Chemins

RAW_PATH = "../data/raw/free tweet export 2.csv"
timestamp = datetime.now().strftime("%Y-%m-%d_%Hh%M")
print(f"{timestamp}")
SILVER_PATH = f"../data/silver/tweets_cleaned_1.csv"
QUALITY_LOG_PATH  = "../quality/quality_log_tweets.csv"


    

2025-09-26_08h11


In [25]:
# 🧽 Fonction de nettoyage
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"@[\w_]+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"["
                  u"\U0001F600-\U0001F64F"
                  u"\U0001F300-\U0001F5FF"
                  u"\U0001F680-\U0001F6FF"
                  u"\U0001F1E0-\U0001F1FF"
                  "]+", "", text, flags=re.UNICODE)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
    

In [26]:
# 📥 Chargement des données brutes
df = pd.read_csv(RAW_PATH)
df.head()
    

,id,created_at,full_text,media,screen_name,name,profile_image_url,user_id,in_reply_to,retweeted_status,...,favorite_count,retweet_count,bookmark_count,quote_count,reply_count,views_count,favorited,retweeted,bookmarked,url
0,1343458257915031553,2020-12-28 08:26:23 +01:00,"💩 à @free parce-que Débit Très instable, … \n\...",[],m_annuel,M Annuel,https://abs.twimg.com/sticky/default_profile_i...,1104790986801250304,NaN,NaN,...,2,1,0,0,1,NaN,False,False,False,https://twitter.com/m_annuel/status/1343458257...
1,1393158240083587075,2021-05-14 12:56:22 +02:00,"RT @free: Retrouvez désormais @ToonamiFR, la c...","[{""type"":""photo"",""url"":""https://t.co/kuAYafYDi...",Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,1.393125e+18,...,0,16,0,0,0,NaN,False,False,False,https://twitter.com/Freebox/status/13931582400...
2,1403337211475546112,2021-06-11 15:03:58 +02:00,"RT @free: A suivre ce soir, le 1er match de l’...","[{""type"":""photo"",""url"":""https://t.co/gMTcYtGdd...",Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,1.403329e+18,...,0,15,0,0,0,NaN,False,False,False,https://twitter.com/Freebox/status/14033372114...
3,1403337257571004417,2021-06-11 15:04:09 +02:00,RT @free: Disponible sur le canal 101 avec les...,[],Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,1.403333e+18,...,0,5,0,0,0,NaN,False,False,False,https://twitter.com/Freebox/status/14033372575...
4,1418550491034882052,2021-07-23 14:36:07 +02:00,« Faites vos premiers pas avec nous ! Découvre...,"[{""type"":""video"",""url"":""https://t.co/YCMv79evb...",Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,NaN,...,31,7,0,3,35,NaN,False,False,False,https://twitter.com/Freebox/status/14185504910...


In [27]:
# 🧼 Application du nettoyage
df["clean_text"] = df["full_text"].apply(clean_text)
df["cleaned_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    

In [28]:
# 🚫 Suppression des comptes officiels
EXCLUDED_ACCOUNTS = [
    "free", "free_1337", "free1337", "groupeiliad", "iliad",
    "free_officiel", "free_official", "groupe_iliad",
    "assistance freebox"
]

df["screen_name"] = df["screen_name"].astype(str).str.lower()
df["name"] = df["name"].astype(str).str.lower()

df_clean = df[
    ~df["screen_name"].isin(EXCLUDED_ACCOUNTS) &
    ~df["name"].isin(EXCLUDED_ACCOUNTS)
].copy()

print(f"✅ {len(df) - len(df_clean)} comptes officiels exclus.")
df_clean[["screen_name", "name"]].drop_duplicates().head()
    

✅ 3324 comptes officiels exclus.


,screen_name,name
0,m_annuel,m annuel
23,mitchelmcpat,mitchelmcpat
29,jnbarrot,jean-noël barrot
30,rgaidot,ɹˈe͡ɪɡɪz
31,ellaverite,𝔼lla 𝕍érité ⚖️ⓩ


In [29]:
# 💾 Export du fichier nettoyé (filtré)
os.makedirs(os.path.dirname(SILVER_PATH), exist_ok=True)
df_clean[["id", "created_at", "screen_name", "full_text", "clean_text", "cleaned_at"]].to_csv(SILVER_PATH, index=False)
print(f"✅ Exporté vers : {SILVER_PATH}")
    

✅ Exporté vers : ../data/silver/tweets_cleaned_1.csv


In [30]:
# 📊 Logging qualité enrichi

n_raw = len(df)
n_cleaned = len(df_clean)
n_delta = n_raw - n_cleaned
nulls = df_clean["full_text"].isna().sum()
pct_nulls = round(nulls / n_cleaned * 100, 2)
avg_length_cleaned = round(df_clean["clean_text"].str.len().mean(), 2)

log_entry = pd.DataFrame([{
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "raw_file": RAW_PATH,
    "export_file": SILVER_PATH,
    "n_total_raw": n_raw,
    "n_total_cleaned": n_cleaned,
    "n_removed_official": n_delta,
    "n_nulls": nulls,
    "pct_nulls": pct_nulls,
    "avg_length_clean_text": avg_length_cleaned
}])

os.makedirs(os.path.dirname(QUALITY_LOG_PATH), exist_ok=True)
if os.path.exists(QUALITY_LOG_PATH):
    print("📈 Ajout au log qualité existant")
    log_entry.to_csv(QUALITY_LOG_PATH, mode='a', header=False, index=False)
else:
    print("🆕 Création du log qualité else")
    log_entry.to_csv(QUALITY_LOG_PATH, index=False)


    

🆕 Création du log qualité else
